# **Objective: To analyze sales patterns, demand behavior, and price sensitivity across products, and derive actionable pricing and revenue strategies using elasticity-based segmentation.**

***dataset used :- online-retail-ii-uci.zip, 2 years retail data of a certain company that have sales accross several countries but 90% from United Kindom. Variables include Quantity, Price variations, Description of product / name of product and some unneccsary columns that removed further in this notebook***

In [0]:
%pip install kaggle


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!kaggle datasets download -d mashlyn/online-retail-ii-uci



Dataset URL: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci
License(s): CC0-1.0
100%|██████████████████████████████████████| 14.5M/14.5M [00:00<00:00, 87.3MB/s]



In [0]:
!unzip online-retail-ii-uci.zip

Archive:  online-retail-ii-uci.zip
  inflating: online_retail_II.csv    


In [0]:
# moving the dataset into base folder for better call

import shutil

shutil.move(
"/Workspace/Users/addysiddiqui7@gmail.com/The Logistic Project/online_retail_II.csv",
"/Workspace/online_retail_II.csv"
)

'/Workspace/online_retail_II.csv'

In [0]:
df = spark.read.csv("/Workspace/online_retail_II.csv", header=True, inferSchema=True)
display(df.limit(2))


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom


In [0]:
display(df.describe())

# total of 144867 rows are in our dataset

summary,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
count,1067371,1067371,1062989,1067371,1067371,824364,1067371
mean,537608.1499316233,28350.201592689715,21848.25,9.9388984711033,4.649387727416074,15324.63850435002,null
stddev,26662.45044690487,17968.479697262945,922.9197780233488,172.7057940767533,123.55305872146253,1697.4644503793093,null
min,489434,10002,DOORMAT UNION JACK GUNS AND ROSES,-80995,-53594.36,12346.0,Australia
max,C581569,m,wrongly sold sets,80995,38970.0,18287.0,West Indies


In [0]:
# removing negative Quanitity columns as the scope of this project is not return analysis.
df = df.filter((df.Quantity > 0) & (df.Price > 0))


In [0]:
display(df.head(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom


In [0]:
dex1 = df.filter(df.Description == "PINK CHERRY LIGHTS")
display(dex1)


# stockcode is identical for every single unit so be dropped further

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489460,79323P,PINK CHERRY LIGHTS,8,2009-12-01T10:46:00.000Z,6.75,16167.0,United Kingdom
489539,79323P,PINK CHERRY LIGHTS,24,2009-12-01T12:18:00.000Z,5.45,15061.0,United Kingdom
489656,79323P,PINK CHERRY LIGHTS,48,2009-12-01T17:28:00.000Z,5.45,17428.0,United Kingdom
489658,79323P,PINK CHERRY LIGHTS,6,2009-12-01T17:31:00.000Z,6.75,15485.0,United Kingdom
489664,79323P,PINK CHERRY LIGHTS,5,2009-12-01T18:03:00.000Z,6.75,14061.0,United Kingdom
489792,79323P,PINK CHERRY LIGHTS,2,2009-12-02T12:06:00.000Z,6.75,14527.0,United Kingdom
489833,79323P,PINK CHERRY LIGHTS,1,2009-12-02T14:09:00.000Z,6.75,14980.0,United Kingdom
489856,79323P,PINK CHERRY LIGHTS,1,2009-12-02T14:36:00.000Z,13.87,null,United Kingdom
489857,79323P,PINK CHERRY LIGHTS,1,2009-12-02T14:43:00.000Z,13.87,null,United Kingdom


In [0]:
#dropping unncessarry columns that don't have any use in scope of this project...

df = df.drop("StockCode","Invoice", "Customer ID")

In [0]:
from pyspark.sql.functions import min, max

display(
    df.select(
        min("InvoiceDate").alias("start_date"),
        max("InvoiceDate").alias("end_date")
    )
)

#dataset start date is 01/12/2009 and end date is 09/12/2009. Which means its a 2 year continuous dataset.

start_date,end_date
2009-12-01T07:45:00.000Z,2011-12-09T12:50:00.000Z


In [0]:
display(df.head(5))

Description,Quantity,InvoiceDate,Price,Country
15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,United Kingdom
PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,United Kingdom
WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,United Kingdom
"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,United Kingdom
STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,United Kingdom


In [0]:
df.write.format("delta").saveAsTable("retail")
# conclusion: there are 5399 total unique products purchased accross 43 countries in span of two years.

## **A quick summary of dataset...**

In [0]:
%sql
SELECT
    COUNT(DISTINCT Description) AS unique_products,
    SUM(Quantity) AS total_quantity_sold,
    SUM(Price) AS total_price,

    (SELECT Description
     FROM retail
     GROUP BY Description
     ORDER BY SUM(Quantity) DESC
     LIMIT 1) AS most_purchased_product,

    (SELECT Description
     FROM retail
     GROUP BY Description
     ORDER BY SUM(Quantity) ASC
     LIMIT 1) AS least_purchased_product

FROM retail

unique_products,total_quantity_sold,total_price,most_purchased_product,least_purchased_product
5399,11420306,4246932.258000009,WORLD WAR 2 GLIDERS ASSTD DESIGNS,CRYSTAL SMALL JEWELLED PHOTOFRAME


### Total of 5399 unique products being purchased.        Total quantity sold 11,420,306 units at a total price of 4,245,932 units. The most purchased product was WORLD WAR 2 GLIDERS and Least purchased was a CRYSTAL SMALL JWELLED PHOTOFRAME.